# Random Forest Hyperparameter Tuning

This notebook **focuses on improving the performance of the Random Forest Model** selected during model building.

Random Forest performed best among the four models tested, with R2 Score of 0.9167

In this notebook we will test different hyperparameter settings to find the best-performing Random Forest model.

### Main Steps:

1. Load the processed dataset
2. Prepare the features and target
3. Split the data into training
4. Create a baseline Random Forest model
5. Understand Random Forest hyperparameters
6. Perform hyperparameter tuning 
7. Train the tuned Random Forest model
8. Evaluate the tuned model
9. Compare the baseline and tuned models
10. Select the final model

### Import Libraries

In [1]:
# Import required libraries for Random Forest Hyperparameter tuning

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

ImportError: No module named 'sklearn.__check_build._check_build'
___________________________________________________________________________
Contents of d:\Programing Tools And Packages\python\Lib\site-packages\sklearn\__check_build:
meson.build               _check_build.cp314-win_amd64.lib_check_build.cp314-win_amd64.pyd
_check_build.pyx          __init__.py               __pycache__
___________________________________________________________________________
It seems that scikit-learn has not been built correctly.

If you have installed scikit-learn from source, please do not forget
to build the package before using it. For detailed instructions, see:
https://scikit-learn.org/dev/developers/advanced_installation.html#building-from-source

If you have used an installer, please check that it is suited for your
Python version, your operating system and your platform.

## Load Processed Dataset

The processed dataset contains the features prepared during data preprocessing and feature engineering.

We will load this dataset and use it to prepare the data for Random Forest hyperparameter tuning.

In [ ]:
# Import pandas
import pandas as pd

# Load the processed dataset
df = pd.read_csv("../Data/03_Processed/mental_health_processed.csv")

# Display dataset shape
print("Dataset Shape:", df.shape)

Dataset Shape: (4998, 77)


## Seperate Features and Target 

The target variable is 'Mental_Health_Score'

All remaining columns are used as input features for the Random Forest model

In [ ]:
# Seperate input features and target variable

X = df.drop("Mental_Health_Score", axis=1)
y = df["Mental_Health_Score"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (4998, 76)
Target Shape: (4998,)


# Train-Test Split

The dataset is divided into training and testing sets using an 80:20 ratio

The training data is used to train and tune the Random Forest Model, while the testing data is kept separate for final evaluation

In [ ]:
# Import train_test_split from sklearn
from sklearn.model_selection import train_test_split


# Split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)


print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)
print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)


Training Features: (3998, 76)
Testing Features: (1000, 76)
Training Target: (3998,)
Testing Target: (1000,)


## Baseline Random Forest Model

Before hyperparameter tuning, we create a baseline Random Forest model using the default parameter settings

The performance of this baseline model will be compared with the tuned Random Forest model to determine whether hyperparameter tuning improves the model

In [ ]:
# Create the baseline Random Forest model 

baseline_rf = RandomForestRegressor(random_state=42)

# Train baseline model
baseline_rf.fit(X_train,y_train)

print("Baseline Random Forest Model Trained Successfully!")

Baseline Random Forest Model Trained Successfully!


## Random Forest Hyperparameters 

A Random Forest Model contains several hyperparameters that control how individual decision trees are created and how the forest learns from the data

Main Hyperparameters we will tune are:
- **'n_estimators'** -> Number of decision trees in the forest
- **'max_depth'** -> Maximum depth of each decision tree
- **'min_samples_split'** -> Minimum number of samples required to split an internal node
- **'min_samples_leaf'** -> Minimun number of samples required in a leaf node
- **'max_features'** -> Number of features considered when looking for the best split

### Why Tune These Parameters?

The default settings are not always optimal for out dataset

By testing different combinations, we can find a configuration that provides better generalization and potentially improves the evaluation metrics of the Random Forest model

Out goal is to improve the model's performance while avoiding overfitting!


In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1,2,4],
    "max_features": ["sqrt", "log2", None]

}

print("Parameter values are ready for tuning")

Parameter values are ready for tuning


In [ ]:
rf_random_search = RandomizedSearchCV(
    estimator = RandomForestRegressor(random_state=42),
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

# estimator → Random Forest model to tune
# param_distributions=param_grid → use the parameter values we just created
# n_iter=20 → test 20 different combinations
# cv=5 → use 5-fold cross-validation to check each combination
# scoring="r2" → use R² to decide which combination performs better
# random_state=42 → gives repeatable results
# n_jobs=-1 → uses available CPU cores to make the search faster
print("Randomized Search setup completed")

Randomized Search setup completed


## Runing Random Search

Random Forest is trained with diffn parameter combinations to find best settings

In [ ]:
rf_random_search.fit(X_train, y_train)

print("Random Forest Hyperparameter tuning completed successfully!")

In [ ]:
# To see which parameter combination performed best, we can check the best_params_ attribute of the RandomizedSearchCV object.

print("Best Parameters: ")
print(rf_random_search.best_params_)


print("\nBest Cross-Validation R2 Score: ")
print(rf_random_search.best_score_)

## Observation 

- Random Forest tuning completed using RandomizedSearchCV with **20 parameter** combinations and **5-fold cross-validn**
- Best combination used *500 trees, 'log2' max features , and no fixed maxim depth*
- Best cross-validation R2 score was approximately **0.9033**
- The tuned model will now be tested on the test data to check its actual performance

## Best Random Forest Model

Best parameters found during tuning are used to create final Random Forest Model

In [ ]:
best_rf = rf_random_search.best_estimator_

print("Best Random Forest Model Created Successfully!")

### Make predictions

In [ ]:
y_pred_tuned = best_rf.predict(X_test)

print("Predictions on Test Set Completed Successfully!")

### Evaluation of tuned RF

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import numpy as np

# Evaluation metric Calculation

mae_tuned = mean_absolute_error(y_test, y_pred_tuned)

rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))

r2_tuned = r2_score(y_test, y_pred_tuned)

In [ ]:
tuned_rf_results = pd.DataFrame({
    "Model": ["Tuned Random Forest"],
    "MAE": [mae_tuned],
    "RMSE": [rmse_tuned],
    "R2 Score": [r2_tuned]
})

tuned_rf_results

## Model Comparison

In [ ]:
model_comparison = pd.DataFrame({
    "Model": ["Random Forest", "Tuned Random Forest"],
    "MAE": [0.277949, mae_tuned],
    "RMSE": [0.385650, rmse_tuned],
    "R2 Score": [0.916690, r2_tuned]
}).set_index("Model")

model_comparison

## Key Insights 

- The Tuned model performed better than original Random Forest model
- MAE **decreased from 0.277949 to 0.267019** which is about 3.9% ↓
- RMSE **decreased from 0.385650 to 0.358643** which is abt 7.0% ↓
- R2 **increased from 0.916690 to 0.927950** which is abt 1.13% ↑
- Hence model explains abt **92.8%** of variations in Mental Health Score

## Further Random Forest Tuning

The first tuning round improved Random Forest Performance 

A wider parameter search is using to check further improvement is possible 

In [ ]:
# Define a wider range of Random Forest parameters for tuning

strong_param_grid = {
    "n_estimators": 
}